# EDA
Exploratory analysis of the two opportunity Excel files and the proposals/responses JSON.

In [1]:
import sys
sys.path.insert(0, '..')

import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

DATA_DIR = Path('../data')
CSV_DIR  = DATA_DIR / 'csv_files'

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)

## 1. Load Raw Data

In [2]:
opps1 = pd.read_excel(CSV_DIR / 'anonymized_opps_1.xlsx', sheet_name='Data')
opps2 = pd.read_excel(CSV_DIR / 'anonymized_opps_2.xlsx', sheet_name='Data')

with open(DATA_DIR / 'proposals_responses.json', encoding='utf-8') as f:
    proposals = json.load(f)

print(f'opps1 shape : {opps1.shape}')
print(f'opps2 shape : {opps2.shape}')
print(f'proposals   : {len(proposals)} entries')
print()
print(f'opps1 columns: {list(opps1.columns)}')
print(f'opps2 columns: {list(opps2.columns)}')

opps1 shape : (8435, 20)
opps2 shape : (8695, 33)
proposals   : 19 entries

opps1 columns: ['(Do Not Modify) Opportunity - Product', '(Do Not Modify) Row Checksum', '(Do Not Modify) Modified On', 'Opportunity ID', 'Opportunity name', 'End client name', 'Opportunity manager', 'Opportunity owner', 'Estimated close date ', 'Status ', 'Probability ', 'Sales stage', 'Opportunity Estimated revenue (Base) (CAD) ', 'Primary territory', 'Service / solution', 'Service / Solution Estimated revenue', 'IP', 'Actual revenue (Base) (CAD)', 'Close date', 'Delivery Territory / Center']
opps2 columns: ['(Do Not Modify) Opportunity', '(Do Not Modify) Row Checksum', '(Do Not Modify) Modified on', 'Opportunity ID', 'Opportunity name', 'End client name', 'Opportunity manager', 'Opportunity owner', 'Primary territory', 'Status', 'Estimated close date', 'Probability', 'Sales stage', 'Total estimated revenue', 'Weighted revenue (Base) (CAD)', 'Comments', 'Is DPSC Required Updated On', 'Close date', 'Actual rev

## 2. Clean & Normalise Column Names
Note: `opps2` has a duplicate `modified_on` column — deduplicated with a numeric suffix.

In [3]:
def clean_cols(df):
    df = df.copy()
    cols = (
        df.columns
          .str.replace(r'\(Do Not Modify\)\s*', '', regex=True)
          .str.strip()
          .str.lower()
          .str.replace(r'[^a-z0-9]+', '_', regex=True)
          .str.strip('_')
    )
    # deduplicate (modified_on appears twice in opps2)
    seen = {}
    new = []
    for c in cols:
        if c in seen:
            seen[c] += 1
            new.append(f'{c}_{seen[c]}')
        else:
            seen[c] = 1
            new.append(c)
    df.columns = new
    return df

o1 = clean_cols(opps1)
o2 = clean_cols(opps2)

print('o1 columns:', list(o1.columns))
print()
print('o2 columns:', list(o2.columns))

o1 columns: ['opportunity_product', 'row_checksum', 'modified_on', 'opportunity_id', 'opportunity_name', 'end_client_name', 'opportunity_manager', 'opportunity_owner', 'estimated_close_date', 'status', 'probability', 'sales_stage', 'opportunity_estimated_revenue_base_cad', 'primary_territory', 'service_solution', 'service_solution_estimated_revenue', 'ip', 'actual_revenue_base_cad', 'close_date', 'delivery_territory_center']

o2 columns: ['opportunity', 'row_checksum', 'modified_on', 'opportunity_id', 'opportunity_name', 'end_client_name', 'opportunity_manager', 'opportunity_owner', 'primary_territory', 'status', 'estimated_close_date', 'probability', 'sales_stage', 'total_estimated_revenue', 'weighted_revenue_base_cad', 'comments', 'is_dpsc_required_updated_on', 'close_date', 'actual_revenue', 'status_reason', 'opportunity_type', 'sales_model', 'project_duration_number_of_months', 'revenue_per_year', 'revenue_per_year_base', 'revenue_start_date', 'created_on', 'current_stage_start_dat

## 3. ID Overlap Between the Two Files

In [4]:
ids1 = set(pd.to_numeric(o1['opportunity_id'], errors='coerce').dropna().astype(int))
ids2 = set(pd.to_numeric(o2['opportunity_id'], errors='coerce').dropna().astype(int))

print(f'IDs only in opps1      : {len(ids1 - ids2)} unique IDs')
print(f'IDs only in opps2      : {len(ids2 - ids1):>5,}')
print(f'IDs in BOTH files      : {len(ids1 & ids2):>5,}')
print(f'Total unique IDs       : {len(ids1 | ids2):>5,}')

# How many rows do those 51 opps1-exclusive IDs correspond to?
o1_only_rows = o1[o1['opportunity_id'].isin(ids1 - ids2)]
print(f'\nRows in opps1 for those 51 exclusive IDs: {len(o1_only_rows)}')
print(f'  (10 IDs appear more than once, creating 11 extra rows → 62 rows, not 51)')

# Column overlap
shared = set(o1.columns) & set(o2.columns)
only1  = set(o1.columns) - set(o2.columns)
only2  = set(o2.columns) - set(o1.columns)
print(f'\nShared columns ({len(shared)}): {sorted(shared)}')
print(f'opps1-only columns ({len(only1)}): {sorted(only1)}')
print(f'opps2-only columns ({len(only2)}): {sorted(only2)}')

IDs only in opps1      : 51 unique IDs
IDs only in opps2      : 1,067
IDs in BOTH files      : 7,628
Total unique IDs       : 8,746

Rows in opps1 for those 51 exclusive IDs: 62
  (10 IDs appear more than once, creating 11 extra rows → 62 rows, not 51)

Shared columns (13): ['close_date', 'end_client_name', 'estimated_close_date', 'modified_on', 'opportunity_id', 'opportunity_manager', 'opportunity_name', 'opportunity_owner', 'primary_territory', 'probability', 'row_checksum', 'sales_stage', 'status']
opps1-only columns (7): ['actual_revenue_base_cad', 'delivery_territory_center', 'ip', 'opportunity_estimated_revenue_base_cad', 'opportunity_product', 'service_solution', 'service_solution_estimated_revenue']
opps2-only columns (20): ['actual_revenue', 'comments', 'created_on', 'current_stage_start_date', 'free_field_text_2', 'is_dpsc_required_updated_on', 'modified_by', 'modified_on_2', 'opportunity', 'opportunity_type', 'project_duration_number_of_months', 'proposal_submission_date', '

## 4. Merge Strategy

**opps2** is the base (richer schema, 33 cols). For the **7,628 overlapping IDs** we left-join to bring columns exclusive to opps1 into opps2.

The **51 unique IDs exclusive to opps1** correspond to **62 rows** (10 IDs appear more than once, creating 11 extra rows); these are appended as-is and kept as duplicates because deduplication logic belongs in a downstream data-cleaning step with business rules.

In [5]:
# Validate whether o2.total_estimated_revenue corresponds to:
# 1. o1 opportunity_estimated_revenue_base_cad
# 2. o1 service_solution_estimated_revenue
# 3. o1 opportunity_estimated_revenue_base_cad + service_solution_estimated_revenue

rev_compare = o1.copy()

rev_compare['opportunity_estimated_revenue_base_cad'] = pd.to_numeric(
    rev_compare['opportunity_estimated_revenue_base_cad'], errors='coerce'
)
rev_compare['service_solution_estimated_revenue'] = pd.to_numeric(
    rev_compare['service_solution_estimated_revenue'], errors='coerce'
)

o2_rev = o2[['opportunity_id', 'total_estimated_revenue']].copy()
o2_rev['total_estimated_revenue'] = pd.to_numeric(
    o2_rev['total_estimated_revenue'], errors='coerce'
)

# Collapse opps1 to opportunity_id level
o1_rev_by_id = (
    rev_compare
    .groupby('opportunity_id')
    .agg(
        o1_opportunity_total=('opportunity_estimated_revenue_base_cad', 'first'),
        o1_service_solution_sum=('service_solution_estimated_revenue', 'sum'),
        o1_service_solution_count=('service_solution_estimated_revenue', 'count'),
        o1_rows=('opportunity_id', 'size'),
    )
    .reset_index()
)

overlap_rev = o2_rev.merge(o1_rev_by_id, on='opportunity_id', how='inner')

overlap_rev['opportunity_plus_service'] = (
    overlap_rev['o1_opportunity_total'].fillna(0)
    + overlap_rev['o1_service_solution_sum'].fillna(0)
)

def almost_equal(a, b):
    return a.round(2).eq(b.round(2))

n = len(overlap_rev)

print(f'Overlapping opportunity IDs: {n:,}')
print()

print(
    'o2 total == o1 opportunity total:',
    f'{almost_equal(overlap_rev["total_estimated_revenue"], overlap_rev["o1_opportunity_total"]).sum():,}',
    f'/ {n:,}'
)

print(
    'o2 total == o1 service solution sum:',
    f'{almost_equal(overlap_rev["total_estimated_revenue"], overlap_rev["o1_service_solution_sum"]).sum():,}',
    f'/ {n:,}'
)

print(
    'o2 total == o1 opportunity total + service sum:',
    f'{almost_equal(overlap_rev["total_estimated_revenue"], overlap_rev["opportunity_plus_service"]).sum():,}',
    f'/ {n:,}'
)

print()
print('Median absolute difference:')
print(
    'o2 total vs o1 opportunity total:',
    (overlap_rev['total_estimated_revenue'] - overlap_rev['o1_opportunity_total']).abs().median()
)
print(
    'o2 total vs opportunity + service:',
    (overlap_rev['total_estimated_revenue'] - overlap_rev['opportunity_plus_service']).abs().median()
)

service_equals_opp = overlap_rev[
    almost_equal(
        overlap_rev['o1_service_solution_sum'],
        overlap_rev['o1_opportunity_total']
    )
]

print()
print(f'IDs where service sum equals opportunity total: {len(service_equals_opp):,}')
print(
    'Among those, o2 total equals opportunity total:',
    almost_equal(
        service_equals_opp['total_estimated_revenue'],
        service_equals_opp['o1_opportunity_total']
    ).sum()
)
print(
    'Among those, o2 total equals opportunity + service:',
    almost_equal(
        service_equals_opp['total_estimated_revenue'],
        service_equals_opp['opportunity_plus_service']
    ).sum()
)

overlap_rev[
    almost_equal(overlap_rev['total_estimated_revenue'], overlap_rev['o1_opportunity_total'])
    & ~almost_equal(overlap_rev['total_estimated_revenue'], overlap_rev['opportunity_plus_service'])
][[
    'opportunity_id',
    'total_estimated_revenue',
    'o1_opportunity_total',
    'o1_service_solution_sum',
    'opportunity_plus_service',
    'o1_service_solution_count',
    'o1_rows',
]].head(10)


Overlapping opportunity IDs: 7,628

o2 total == o1 opportunity total: 7,522 / 7,628
o2 total == o1 service solution sum: 4,176 / 7,628
o2 total == o1 opportunity total + service sum: 2,946 / 7,628

Median absolute difference:
o2 total vs o1 opportunity total: 0.0
o2 total vs opportunity + service: 1100.0

IDs where service sum equals opportunity total: 4,105
Among those, o2 total equals opportunity total: 4104
Among those, o2 total equals opportunity + service: 10


,opportunity_id,total_estimated_revenue,o1_opportunity_total,o1_service_solution_sum,opportunity_plus_service,o1_service_solution_count,o1_rows
0,10501370,135000000.0,135000000.0,135000000.0,270000000.0,1,1
1,10529441,112000000.0,112000000.0,112000000.0,224000000.0,1,1
2,10621656,50250000.0,50250000.0,50250000.0,100500000.0,3,3
3,10634654,45189000.0,45189000.0,45189000.0,90378000.0,1,1
5,10567978,33000000.0,33000000.0,33000000.0,66000000.0,1,1
6,10607306,30605257.0,30605257.0,30605257.0,61210514.0,1,1
7,10661703,30134000.0,30134000.0,30134000.0,60268000.0,1,1
9,10586584,30000000.0,30000000.0,30000000.0,60000000.0,3,3
10,10541507,29700000.0,29700000.0,29700000.0,59400000.0,1,1
13,10624718,25000000.0,25000000.0,25000000.0,50000000.0,1,1


In [6]:
# Rename opps1 revenue columns to match opps2 naming.

o1_aligned = o1.rename(columns={
    'opportunity_estimated_revenue_base_cad': 'total_estimated_revenue',
    'actual_revenue_base_cad'               : 'actual_revenue',
})

# Columns present in opps1 but not opps2 — bring them in for shared rows
o1_supplement_cols = ['opportunity_id'] + [c for c in o1_aligned.columns if c not in o2.columns]
print('Supplemental columns from opps1:', o1_supplement_cols)

# Left-join the supplement onto opps2
df = o2.merge(
    o1_aligned[o1_supplement_cols].drop_duplicates(subset=['opportunity_id']),
    on='opportunity_id',
    how='left',
)

# Append rows exclusive to opps1 (51 unique IDs / 62 rows)
o1_exclusive = o1_aligned[o1_aligned['opportunity_id'].isin(ids1 - ids2)]
df = pd.concat([df, o1_exclusive], ignore_index=True)

print(f'Merged shape          : {df.shape}')
print(f'Unique opportunity_id : {df["opportunity_id"].nunique():,}')
print(f'Duplicate rows        : {df.duplicated(subset=["opportunity_id"]).sum()}')
df.head(3)

Supplemental columns from opps1: ['opportunity_id', 'opportunity_product', 'service_solution', 'service_solution_estimated_revenue', 'ip', 'delivery_territory_center']
Merged shape          : (8757, 38)
Unique opportunity_id : 8,746
Duplicate rows        : 11


,opportunity,row_checksum,modified_on,opportunity_id,opportunity_name,end_client_name,opportunity_manager,opportunity_owner,primary_territory,status,estimated_close_date,probability,sales_stage,total_estimated_revenue,weighted_revenue_base_cad,comments,is_dpsc_required_updated_on,close_date,actual_revenue,status_reason,opportunity_type,sales_model,project_duration_number_of_months,revenue_per_year,revenue_per_year_base,revenue_start_date,created_on,current_stage_start_date,modified_by,modified_on_2,proposal_submission_date,rfp_release_date,free_field_text_2,opportunity_product,service_solution,service_solution_estimated_revenue,ip,delivery_territory_center
0,e8751e0a-ae2c-e811-8111-005056010712,Qso+SyGwSnpXSAPcpcxWJlHkXG65GoV40Kq7VAg6vkH2dOzFLIKPHqCL...,2021-04-30 05:20:47,10007954,Harbor Technology - Managed IT Services (Phase 2),Harbor Technology Group (2),Jordan Cromwell,Cameron Foxworth,CAN ATL Atlantic Metro,Closed,2020-09-27,10.0,0-Lead/Suspect,200000000.0,20000000.0,NaN,NaT,2019-05-13,0.0,Cancelled By CGI,Existing Client - New Business,Outsourcing,120.0,20000000.0,20000000.0,NaT,2018-03-20 23:17:45,NaT,TFS Build Service,2021-04-30 02:20:47,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN
1,cc7a89df-71b5-ee11-9cc4-005056852fa8,yEY/V79nofLl+EDQUBMpc0VRah1wwM7PWpz5wYt8VzITfp2l58ZEucjB...,2024-08-07 14:28:47,10501370,Unity Systems - Staff Augmentation,Unity Systems Inc,Emery Upton,Lee Northcott,CAN ATL Ntl Svcs Credit Union,Closed,2024-07-31,100.0,4-Proposal,135000000.0,135000000.0,Closing this opportunity. Replacing with a list of Celer...,2024-03-12 16:32:00,2024-08-07,0.0,Duplicated,New Client - New Business,Outsourcing,18.0,90000000.0,90000000.0,2024-07-31,2024-01-17 15:51:59,2024-07-05 17:49:28,Robert Power,2024-08-07 11:28:47,NaT,NaT,NaN,295c84a7-08db-ee11-9cc8-005056858efc,Application Management,135000000.0,No,NaN
2,ad804cad-6918-ef11-9cca-005056852fa8,WDThnehnXCrfw1xakuywJn/l5XbaCw96LsBlGEGk2r0AQ5ePoQXhiaav...,2025-02-26 20:17:38,10529441,Apex Consulting - DevOps Transformation (Phase 4),Apex Consulting Group (2),Emery Upton,Lee Northcott,CAN ATL Ntl Svcs Credit Union,Closed,2025-06-30,0.0,0-Lead/Suspect,112000000.0,0.0,NaN,2024-03-12 16:32:00,2025-02-26,0.0,Duplicated,New Client - New Business,Outsourcing,18.0,74666667.0,74666667.0,NaT,2024-05-22 15:32:46,2024-05-22 15:32:46,SYSTEM,2025-02-26 16:17:38,NaT,NaT,NaN,e4804cad-6918-ef11-9cca-005056852fa8,Application Management,112000000.0,No,NaN


## 5. Parse Types & Inspect Missing Values

In [7]:
date_cols = [
    'estimated_close_date', 'close_date', 'created_on',
    'proposal_submission_date', 'rfp_release_date',
    'revenue_start_date', 'current_stage_start_date',
]
for c in date_cols:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], errors='coerce')

num_cols = [
    'total_estimated_revenue', 'actual_revenue', 'probability',
    'weighted_revenue_base_cad', 'project_duration_number_of_months',
    'revenue_per_year',
]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

print('Types set.')

Types set.


In [8]:
null_pct = (df.isnull().mean() * 100).round(1).sort_values(ascending=False)
print('NULL RATES (%):')
print(null_pct[null_pct > 0].to_string())

NULL RATES (%):
free_field_text_2                     98.8
delivery_territory_center             88.7
rfp_release_date                      85.2
proposal_submission_date              84.3
is_dpsc_required_updated_on           84.0
comments                              76.0
service_solution_estimated_revenue    44.1
revenue_start_date                    12.6
ip                                    12.2
service_solution                      12.2
opportunity_product                   12.2
current_stage_start_date              12.0
project_duration_number_of_months      9.9
close_date                             4.3
actual_revenue                         4.3
opportunity_type                       4.2
weighted_revenue_base_cad              1.0
probability                            0.9
created_on                             0.7
revenue_per_year                       0.7
modified_by                            0.7
modified_on_2                          0.7
revenue_per_year_base                 

In [9]:
fig = px.bar(
    null_pct[null_pct > 0].reset_index(),
    x='index', y=0,
    labels={'index': 'Column', 0: '% Missing'},
    title='Missing Values by Column (%)',
    color=0,
    color_continuous_scale='reds',
    height=420
)
fig.update_xaxes(tickangle=45)
fig.update_coloraxes(showscale=False)
fig.show()

In [10]:
# Raw opps1 null rates for supplemental columns
raw_o1_cols = [
    'service_solution',
    'opportunity_product',
    'ip',
    'service_solution_estimated_revenue',
    'delivery_territory_center',
]

raw_o1_nulls = (
    o1[raw_o1_cols]
    .isnull()
    .mean()
    .mul(100)
    .round(1)
    .sort_values(ascending=False)
)

print('Raw opps1 null rates (%)')
print(raw_o1_nulls.to_string())

print()
print(f'opps1 unique IDs                 : {o1["opportunity_id"].nunique():,}')
print(f'total unique IDs across files    : {len(ids1 | ids2):,}')
print(f'opps1 coverage of total unique IDs: {100 * o1["opportunity_id"].nunique() / len(ids1 | ids2):.1f}%')
print(f'overlap IDs covered by opps1      : {len(ids1 & ids2):,}')
print(f'overlap share of total unique IDs : {100 * len(ids1 & ids2) / len(ids1 | ids2):.1f}%')

Raw opps1 null rates (%)
delivery_territory_center             86.1
service_solution_estimated_revenue    34.9
service_solution                       0.0
opportunity_product                    0.0
ip                                     0.0

opps1 unique IDs                 : 7,679
total unique IDs across files    : 8,746
opps1 coverage of total unique IDs: 87.8%
overlap IDs covered by opps1      : 7,628
overlap share of total unique IDs : 87.2%


**Note on null rates:** `service_solution`, `opportunity_product`, and `ip`  are 0% missing in raw opps1 (which covers ~88% of total IDs via the overlap), so the merged null rate is driven entirely by how many of those rows had matches in opps2. The columns still have limited capacity-model value because opps2 (the richer, longer-running dataset) never populated them — interpret their completeness at the opps1-origin level only.

Columns that are sparse and should be excluded from the capacity model:
- `proposal_submission_date` (~84% missing across both files)
- `rfp_release_date` (~85% missing)
- `comments` (~76% missing)
- `free_field_text_2` (~99% missing)

## 6. Status & Pipeline Stage Distribution

In [11]:
print('=== STATUS ===')
print(df['status'].value_counts(dropna=False).to_string())
print()
print('=== SALES STAGE ===')
print(df['sales_stage'].value_counts(dropna=False).to_string())

=== STATUS ===
status
Won       5290
Closed    3093
Open       374

=== SALES STAGE ===
sales_stage
6-Negotiation&Signature    5425
0-Lead/Suspect             1027
5-Client Decision           813
1-Identification            485
2-Qualification             390
4-Proposal                  315
3-Bid Planning              302


In [12]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Status Distribution', 'Sales Stage Distribution'])

sc  = df['status'].value_counts(dropna=False)
stg = df['sales_stage'].value_counts(dropna=False)

fig.add_trace(go.Bar(x=sc.index.astype(str),  y=sc.values,  name='Status'), row=1, col=1)
fig.add_trace(go.Bar(x=stg.index.astype(str), y=stg.values, name='Stage'),  row=1, col=2)

fig.update_layout(height=380, showlegend=False, title_text='Opportunity Status & Stage')
fig.show()

In [13]:
# Critical: how many rows are truly "open"?
open_mask = df['status'].str.lower().str.contains('open', na=False)
print(f'Open opportunities : {open_mask.sum():,} / {len(df):,} ({100*open_mask.mean():.1f}%)')
print()
print('status_reason breakdown:')
print(df['status_reason'].value_counts(dropna=False).to_string())

Open opportunities : 374 / 8,757 (4.3%)

status_reason breakdown:
status_reason
Won                                                 5275
Cancelled By CGI                                     871
Cancelled by Customer                                583
Cancelled No Bid Decision                            463
Duplicated                                           461
Open                                                 343
LOST-Unknown/Other                                   262
LOST-Expertise                                       226
LOST-Price                                            62
NaN                                                   62
LOST-Solution                                         50
LOST-Experience                                       46
LOST-Speed/Avail. to Deliver                          27
LOST-Relationship                                     13
Cancelled No Bid Decision due to Client Feedback       6
LOST-Quality                                           4
Cancelle

**Critical finding:** Only 374 open opportunities (4.3%). The dataset is overwhelmingly historical. A real-time capacity view based solely on `status = Open` will be very sparse — see section 10 for the extended load definition.

## 7. Revenue Distribution

In [14]:
rev     = df['total_estimated_revenue'].dropna()
rev_pos = rev[rev > 0]
print(f'Revenue n={len(rev_pos):,}')
print(rev_pos.describe().apply(lambda x: f'${x:>15,.0f}'))

Revenue n=8,462
count    $          8,462
mean     $        424,361
std      $      3,926,485
min      $              0
25%      $          2,940
50%      $         24,420
75%      $        100,000
max      $    200,000,000
Name: total_estimated_revenue, dtype: str


In [15]:
fig = px.histogram(
    np.log10(rev_pos.clip(lower=1)),
    nbins=60,
    title='log10(Estimated Revenue) Distribution',
    labels={'value': 'log10(CAD)'},
    height=350
)
fig.show()

**Finding:** Revenue is extremely right-skewed (median $24K, max $200M). Log-scale normalization is required for the capacity score.

## 8. Probability Distribution

In [16]:
prob = df['probability'].dropna()
n    = len(prob)

cnt_0    = (prob == 0).sum()
cnt_100  = (prob == 100).sum()
cnt_mid  = ((prob > 0) & (prob < 100)).sum()

print(f'n                  : {n:,}')
print(f'== 0               : {cnt_0:,}  ({100*cnt_0/n:.1f}%)')
print(f'== 100             : {cnt_100:,}  ({100*cnt_100/n:.1f}%)')
print(f'intermediate (1-99): {cnt_mid:,}  ({100*cnt_mid/n:.1f}%)')
print()
print(prob.describe().round(1))

fig = px.histogram(prob, nbins=20,
                   title='Probability Distribution (36.9% intermediate values)',
                   labels={'value': 'Probability (%)'},
                   height=320)
fig.show()

n                  : 8,680
== 0               : 139  (1.6%)
== 100             : 5,337  (61.5%)
intermediate (1-99): 3,204  (36.9%)

count    8680.0
mean       73.2
std        36.5
min         0.0
25%        50.0
50%       100.0
75%       100.0
max       100.0
Name: probability, dtype: float64


**Finding:** Probability is skewed heavily toward 100 (61.5%), but 36.9% of values are intermediate (1–99). It is not effectively binary. The 100-heavy skew reflects Won deals (confirmed by `status_reason`). For capacity purposes:
- Use `status` / `status_reason` as the primary won/loss signal, not probability.
- Probability retains some use as a pipeline confidence weight for open deals.

## 9. Project Duration

In [17]:
dur = df['project_duration_number_of_months'].dropna()
dur = dur[(dur > 0) & (dur < 300)]
print(f'n={len(dur):,}')
print(dur.describe().round(1))

fig = px.histogram(dur, nbins=40,
                   title='Project Duration (months)',
                   labels={'value': 'Months'},
                   height=320)
fig.show()

n=7,851
count    7851.0
mean       10.3
std        17.3
min         1.0
25%         2.0
50%         4.0
75%        11.0
max       120.0
Name: project_duration_number_of_months, dtype: float64


**Finding:** Median 4 months; 75th pct 11 months. `revenue_start_date + project_duration` is a viable proxy for ongoing delivery commitments beyond just the open pipeline.

## 10. Director (Opportunity Manager) Analysis

In [18]:
n_mgrs   = df['opportunity_manager'].nunique()
n_owners = df['opportunity_owner'].nunique()
same     = (df['opportunity_manager'].str.strip() == df['opportunity_owner'].str.strip()).sum()

print(f'Unique opportunity_manager : {n_mgrs}')
print(f'Unique opportunity_owner   : {n_owners}')
print(f'manager == owner           : {same:,} rows ({100*same/len(df):.1f}%)')
print('  → manager and owner are always distinct roles')

Unique opportunity_manager : 68
Unique opportunity_owner   : 24
manager == owner           : 0 rows (0.0%)
  → manager and owner are always distinct roles


In [19]:
print('All opportunity_owner values (likely the senior directors, n=24):')
print(df['opportunity_owner'].value_counts(dropna=False).to_string())

All opportunity_owner values (likely the senior directors, n=24):
opportunity_owner
Lee Northcott       4709
Cameron Foxworth    3641
Zara Fairfield        58
Logan Jameson         41
Hadley Randolph       32
Indira Foxworth       31
Jordan Jameson        31
Blake Vickers         30
Emery Jameson         29
Logan Easton          27
Fern Westwood         20
Remy Underhill        19
Briar Quigley         17
Blake Radford         15
Avery Foxworth        14
Zion Kirkland         12
Oakley Dalton         11
Cameron Brennan        7
Morgan Kirkland        5
Val Iverson            2
Peyton Sterling        2
Emery Iverson          2
Quinn Calloway         1
Shawn Yamamoto         1


In [20]:
mgr_total = df.groupby('opportunity_manager').size().sort_values(ascending=False)
print('Top 20 opportunity_manager by all-time opp count:')
print(mgr_total.head(20).to_string())

Top 20 opportunity_manager by all-time opp count:
opportunity_manager
Parker Calloway     981
Uma Iverson         787
Evan Westwood       559
Zara Mercer         559
Kerry Jameson       556
Gale Randolph       499
Blake Brennan       417
Finley Foxworth     375
Yori Osborne        359
Yael Fairfield      311
Peyton Underhill    260
Finley Yamamoto     233
Parker Sterling     221
Gray Dalton         209
Noel Dalton         182
Alex Radford        180
Jordan Cromwell     156
Remy Randolph       148
Morgan Osborne      146
Noel Easton         140


In [21]:
fig = px.bar(
    mgr_total.head(30).reset_index(),
    x='opportunity_manager', y=0,
    title='Top 30 Managers — All-Time Opportunity Count',
    labels={'opportunity_manager': 'Manager', 0: 'Count'},
    height=400
)
fig.update_xaxes(tickangle=50)
fig.show()

In [ ]:
# Active deals and weighted pipeline per manager
# X-axis: Number of currently open opportunities assigned to that manager.
# Y-axis: Total weighted revenue for that manager’s open opportunities:
# weighted_revenue = probability / 100 * total_estimated_revenue

active = df[open_mask].copy()
active_per_mgr = active.groupby('opportunity_manager').size().sort_values(ascending=False)

df['weighted_rev_calc'] = (df['probability'].fillna(0) / 100) * df['total_estimated_revenue'].fillna(0)
weighted_pipeline = (
    df[open_mask]
    .groupby('opportunity_manager')['weighted_rev_calc']
    .sum()
    .sort_values(ascending=False)
)

mgr_stats = pd.DataFrame({
    'active_deals'      : active_per_mgr,
    'weighted_pipeline' : weighted_pipeline,
}).dropna()

fig = px.scatter(
    mgr_stats.reset_index(),
    x='active_deals', y='weighted_pipeline',
    text='opportunity_manager',
    title='Manager — Open Deal Count vs Weighted Pipeline',
    labels={'active_deals': '# Open Deals', 'weighted_pipeline': 'Weighted Pipeline (CAD)'},
    height=500
)
fig.update_traces(textposition='top center', textfont_size=9)
fig.show()

**Finding:** Manager and Owner are always distinct (0% overlap). 68 managers vs. 24 owners — owners are likely the senior directors. Dashboard primary unit = owner; manager breakdown as drill-down.

## 11. Historical Baseline — Quarterly Concurrent Deal Load

In [23]:
# Derive range from the data itself so new records are always included
data_start = df['created_on'].dropna().min()
data_end   = df['created_on'].dropna().max()

q_start_period = pd.Period(data_start, freq='Q')
q_end_period   = pd.Period(data_end,   freq='Q')

print(f'Data runs: {data_start.date()} → {data_end.date()}')
print(f'Quarter range: {q_start_period} → {q_end_period}')

quarters = pd.period_range(q_start_period, q_end_period, freq='Q')

Data runs: 2018-03-12 → 2026-04-06
Quarter range: 2018Q1 → 2026Q2


In [24]:
rows = []
for q in quarters:
    qs = q.start_time
    qe = q.end_time
    mask = (
        df['created_on'].notna() &
        (df['created_on'] <= qe) &
        (df['close_date'].isna() | (df['close_date'] >= qs))
    )
    per_mgr = df[mask].groupby('opportunity_manager').size().reset_index(name='deal_count')
    per_mgr['quarter'] = str(q)
    rows.append(per_mgr)

timeline = pd.concat(rows, ignore_index=True)
print(f'Timeline rows: {len(timeline):,}  covering {len(quarters)} quarters')

Timeline rows: 707  covering 34 quarters


In [25]:
baseline = timeline.groupby('opportunity_manager')['deal_count'].agg(
    hist_mean='mean',
    hist_std='std',
    hist_max='max',
    quarters_seen='count'
).round(2).sort_values('hist_mean', ascending=False)

print(f'Managers with <4 quarters history (unreliable baseline): {(baseline["quarters_seen"] < 4).sum()}')
print(f'Managers with ≥4 quarters history: {(baseline["quarters_seen"] >= 4).sum()}')
print()
print('Top 15 managers by historical mean concurrent deal count:')
print(baseline.head(15).to_string())

Managers with <4 quarters history (unreliable baseline): 17
Managers with ≥4 quarters history: 50

Top 15 managers by historical mean concurrent deal count:
                     hist_mean  hist_std  hist_max  quarters_seen
opportunity_manager                                              
Yori Osborne            105.75    139.17       313              4
Evan Westwood            79.20     77.93       238             10
Xander Brennan           45.33     18.01        66              3
Parker Calloway          44.82     16.80        91             22
Uma Iverson              43.06     39.96       174             34
Peyton Underhill         40.27     21.94        67             11
Zara Mercer              39.77     32.54        81             22
Finley Foxworth          38.67     90.10       352             15
Kerry Jameson            38.59     41.23       166             34
Noel Easton              33.29     18.24        54              7
Alex Radford             33.14     65.02       180 

In [26]:
top5 = mgr_total.head(5).index.tolist()

fig = px.line(
    timeline[timeline['opportunity_manager'].isin(top5)],
    x='quarter', y='deal_count',
    color='opportunity_manager',
    title='Concurrent Deal Count Over Time — Top 5 Managers (full date range)',
    labels={'deal_count': 'Concurrent Deals', 'quarter': 'Quarter'},
    height=420
)
fig.show()

## 12. Territory, Opportunity Type & Sales Model

In [27]:
print('=== PRIMARY TERRITORY ===')
print(df['primary_territory'].value_counts(dropna=False).to_string())

print('\n=== OPPORTUNITY TYPE ===')
print(df['opportunity_type'].value_counts(dropna=False).to_string())

print('\n=== SALES MODEL ===')
print(df['sales_model'].value_counts(dropna=False).to_string())

=== PRIMARY TERRITORY ===
primary_territory
CAN ATL Ntl Svcs Credit Union    4889
CAN ATL Atlantic Metro           3804
CAN Atlantic                       62
CAN ATL Atlantic GDC                2

=== OPPORTUNITY TYPE ===
opportunity_type
Existing Client - Add-on          4192
Existing Client - New Business    3015
Existing Client - Renewal          666
New Client - New Business          512
NaN                                372

=== SALES MODEL ===
sales_model
SI&C           6033
Outsourcing    2662
NaN              62


**Finding:** Data is scoped to Atlantic Canada only — not pan-Canada as the problem statement describes. Clarify scope with CGI before final delivery.

## 13. Proposals / Responses JSON

In [28]:
def approx_tokens(lines):
    """Word-count estimate only (~4/3 words→tokens). Use tiktoken before
    finalising chunk counts against the actual cl100k_base tokenizer."""
    return len(' '.join(lines).split()) * 4 // 3

rows_p = []
for title, entry in proposals.items():
    p = entry['proposal']
    prop_toks = approx_tokens(p.get('content', []))
    resp_toks = sum(
        approx_tokens(v.get('content', []))
        for v in p.get('proposal_response', {}).values()
    )
    rows_p.append({
        'title'          : title[:65],
        'proposal_tokens': prop_toks,
        'response_tokens': resp_toks,
        'total_tokens'   : prop_toks + resp_toks,
    })

pf = pd.DataFrame(rows_p).sort_values('proposal_tokens', ascending=False)
pf

,title,proposal_tokens,response_tokens,total_tokens
13,4. Courts Case Management System,35825,4522,40347
8,17. 107450 MG - RFP - Reporting and Data Analytics Solution,22354,21713,44067
9,18. 105668 SY - RFP for Cybersecurity Operations and Ser...,20958,38746,59704
2,11. HPEI-1382 - Health PEI Data Roadmap,20142,148169,168311
14,5. Salesforce CXM Partner,17520,24966,42486
18,9. SR-25-FY22 - Professional Information Technology Serv...,17496,16989,34485
10,19. 2025-19 RFP Agile Delivery Services,16113,22850,38963
17,8. RFSQ OPOT IT Resources,15818,1474,17292
1,10. SR-18-FY26 - Microsoft Centre of Excellence Partner,13321,12844,26165
15,6. Managed Detection and Response - RFP 8026125-26,12980,3364,16344


In [29]:
# OpenAI embeddings API limit is 8192 tokens per input
# https://platform.openai.com/docs/api-reference/embeddings/create
EMB_LIMIT = 8_192

need_chunk = pf[pf['proposal_tokens'] > EMB_LIMIT]
print(f'Embedding limit (text-embedding-3-small) : {EMB_LIMIT:,} tokens')
print(f'Proposals exceeding limit (word estimate): {len(need_chunk)} / {len(pf)}')
print()
print('NOTE: approx_tokens() uses a word-count heuristic.')
print('Verify with tiktoken (cl100k_base) before finalising chunk sizes.')
print()
print(need_chunk[['title', 'proposal_tokens']].to_string(index=False))

Embedding limit (text-embedding-3-small) : 8,192 tokens
Proposals exceeding limit (word estimate): 15 / 19

NOTE: approx_tokens() uses a word-count heuristic.
Verify with tiktoken (cl100k_base) before finalising chunk sizes.

                                                            title  proposal_tokens
                                 4. Courts Case Management System            35825
      17. 107450 MG - RFP - Reporting and Data Analytics Solution            22354
    18. 105668 SY - RFP for Cybersecurity Operations and Services            20958
                          11. HPEI-1382 - Health PEI Data Roadmap            20142
                                        5. Salesforce CXM Partner            17520
     9. SR-25-FY22 - Professional Information Technology Services            17496
                          19. 2025-19 RFP Agile Delivery Services            16113
                                        8. RFSQ OPOT IT Resources            15818
          10. SR-18-FY26 - 

In [30]:
fig = px.bar(
    pf,
    x='title', y=['proposal_tokens', 'response_tokens'],
    barmode='stack',
    title='Approx Token Length per Proposal (word-count estimate)',
    labels={'value': 'Approx Tokens', 'title': ''},
    height=430
)
fig.add_hline(y=EMB_LIMIT, line_dash='dash', line_color='red',
              annotation_text='8192 embedding limit')
fig.update_xaxes(tickangle=50)
fig.show()

**Finding:** 15 of 19 proposals exceed the 8,192-token embedding limit (word-count estimate). Chunking with overlap is mandatory. Token counts should be verified with `tiktoken` (cl100k_base tokenizer) before finalizing chunk sizes, as the word-count heuristic can be off by ±20%.

## 14. EDA Summary

In [31]:
print('=' * 65)
print('DATA SHAPE')
print('=' * 65)
print(f'  Merged rows                     : {len(df):,}')
print(f'  Unique opportunity IDs          : {df["opportunity_id"].nunique():,}')
print(f'  Duplicate rows (opps1 multi-row): {df.duplicated(subset=["opportunity_id"]).sum()}')
print(f'  Open (active) opportunities     : {open_mask.sum():,} ({100*open_mask.mean():.1f}%)')
print(f'  Unique opportunity_manager      : {df["opportunity_manager"].nunique()}')
print(f'  Unique opportunity_owner        : {df["opportunity_owner"].nunique()}')

print()
print('=' * 65)
print('GENUINELY SPARSE COLUMNS (drop from capacity model)')
print('=' * 65)
genuinely_sparse = [
    'proposal_submission_date', 'rfp_release_date',
    'comments', 'free_field_text_2', 'is_dpsc_required_updated_on'
]
for c in genuinely_sparse:
    if c in df.columns:
        print(f'  {c:<45}: {null_pct.get(c, 0):.1f}% missing')

print()
print('=' * 65)
print('PROBABILITY')
print('=' * 65)
print(f'  Skewed high (61.5% == 100), but 36.9% intermediate')
print(f'  Use status/status_reason for win/loss; probability')
print(f'  retains value as pipeline confidence weight for open deals')

print()
print('=' * 65)
print('PROPOSALS')
print('=' * 65)
print(f'  {len(pf)} total  |  {len(need_chunk)} exceed 8192-token limit (word estimate)')
print(f'  Largest: {pf["proposal_tokens"].max():,} tokens — verify with tiktoken')

print()
print('=' * 65)
print('SCOPE NOTE')
print('=' * 65)
print(f'  Geography: Atlantic Canada only')
print(f'  Date range: {data_start.date()} → {data_end.date()}')
print(f'  Manager vs Owner: always distinct (68 managers / 24 owners)')

DATA SHAPE
  Merged rows                     : 8,757
  Unique opportunity IDs          : 8,746
  Duplicate rows (opps1 multi-row): 11
  Open (active) opportunities     : 374 (4.3%)
  Unique opportunity_manager      : 68
  Unique opportunity_owner        : 24

GENUINELY SPARSE COLUMNS (drop from capacity model)
  proposal_submission_date                     : 84.3% missing
  rfp_release_date                             : 85.2% missing
  comments                                     : 76.0% missing
  free_field_text_2                            : 98.8% missing
  is_dpsc_required_updated_on                  : 84.0% missing

PROBABILITY
  Skewed high (61.5% == 100), but 36.9% intermediate
  Use status/status_reason for win/loss; probability
  retains value as pipeline confidence weight for open deals

PROPOSALS
  19 total  |  15 exceed 8192-token limit (word estimate)
  Largest: 35,825 tokens — verify with tiktoken

SCOPE NOTE
  Geography: Atlantic Canada only
  Date range: 2018-03-12 → 202